# A 3D scene of one block of Brussels from Panoramax street photos

This notebook downloads the open street photos of one block (Panoramax, CC BY-SA 4.0), works out where each photo was taken (COLMAP), and trains a
Gaussian splatting scene from them (gsplat) on the free GPU of Colab. The result is a `.ply` file that you can look at in the browser at
https://playcanvas.com/supersplat/editor (drag the file in).

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Then Runtime > Run all. It takes roughly 30 to 90 minutes; the slow parts are the photo
matching and the training. Nothing has been tested on a real run yet: if a cell stops with an error, copy the error to whoever gave you this notebook.

**What can go wrong:** the photos' positions are only good to about 5 m, so COLMAP may fail to register many of them. The cell after COLMAP says how many it registered.
If it is under about 60% of the photos, the scene will be poor.

In [ ]:
import subprocess, os, json, math, shutil, sys, time
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "NO GPU: Runtime > Change runtime type > T4 GPU")
LAT, LON = 50.8467, 4.3525      # the block: the Grand-Place of Brussels (change it to any place)
RADIUS = 90                     # metres around it
MAX_PHOTOS = 220                # photos (more is better but slower)
os.makedirs("scene/images", exist_ok=True)

In [ ]:
%%bash
set -e
apt-get -qq update > /dev/null
apt-get -qq install -y colmap xvfb > /dev/null
pip -q install requests pillow numpy
[ -d gsplat ] || git clone -q https://github.com/nerfstudio-project/gsplat.git
pip -q install gsplat
pip -q install -r gsplat/examples/requirements.txt
colmap -h | head -2

In [ ]:
# the photos of the block: Panoramax has a search API with no key
import requests, io
from PIL import Image
d = 0.0011
bbox = f"{LON - d*1.6:.6f},{LAT - d:.6f},{LON + d*1.6:.6f},{LAT + d:.6f}"
r = requests.get("https://api.panoramax.xyz/api/search", params={"bbox": bbox, "limit": 1000}, headers={"User-Agent": "photoscan-notebook/0.1"}, timeout=60)
feats = r.json().get("features", [])
print(len(feats), "photos found in the box")
mlat, mlon = 110540.0, 111320 * math.cos(math.radians(LAT))
cands = []
for f in feats:
    p = f["properties"]; lon, lat = f["geometry"]["coordinates"][:2]
    io_ = p.get("pers:interior_orientation") or {}
    if (io_.get("field_of_view") or 70) >= 170: continue        # 360 degree pictures are left out
    dist = math.hypot((lon - LON) * mlon, (lat - LAT) * mlat)
    if dist <= RADIUS and "sd" in f["assets"]: cands.append((dist, f))
cands.sort(key=lambda t: t[0])
step = max(1, len(cands) // MAX_PHOTOS)
chosen = [f for _, f in cands][::step][:MAX_PHOTOS]
print(len(cands), "within", RADIUS, "m, using", len(chosen))
for i, f in enumerate(chosen):
    try:
        b = requests.get(f["assets"]["sd"]["href"], timeout=60, headers={"User-Agent": "photoscan-notebook/0.1"}).content
        Image.open(io.BytesIO(b)).convert("RGB").save(f"scene/images/{i:04d}.jpg", quality=92)
    except Exception as e:
        print("photo", i, "not loaded:", e)
print(len(os.listdir("scene/images")), "photos saved")
print("Photos: Panoramax contributors, CC BY-SA 4.0 (https://panoramax.fr)")

In [ ]:
%%bash
set -e
cd scene
rm -rf colmap.db sparse; mkdir -p sparse
export QT_QPA_PLATFORM=offscreen
colmap feature_extractor --database_path colmap.db --image_path images --ImageReader.single_camera 0 --ImageReader.camera_model SIMPLE_RADIAL --SiftExtraction.use_gpu 0 --SiftExtraction.max_image_size 1600
colmap exhaustive_matcher --database_path colmap.db --SiftMatching.use_gpu 0
colmap mapper --database_path colmap.db --image_path images --output_path sparse
ls sparse

In [ ]:
# how many photos did COLMAP place? (this decides if the rest is worth doing)
import glob
models = sorted(glob.glob("scene/sparse/*"))
best = None
for m in models:
    out = subprocess.run(["colmap", "model_analyzer", "--path", m], capture_output=True, text=True)
    txt = out.stdout + out.stderr
    reg = [int(l.split(":")[-1]) for l in txt.splitlines() if "Registered images" in l]
    print(m, "registered images:", reg[0] if reg else "?")
    if reg and (best is None or reg[0] > best[1]): best = (m, reg[0])
total = len(os.listdir("scene/images"))
print(f"best model: {best}, of {total} photos")
if best:
    if best[0] != "scene/sparse/0":
        shutil.rmtree("scene/sparse/0", ignore_errors=True); shutil.copytree(best[0], "scene/sparse/0")
    print("good enough" if best[1] >= 0.6 * total else "LOW: less than 60% placed, the scene will be poor")

In [ ]:
%%bash
set -e
cd gsplat/examples
python simple_trainer.py default --data_dir ../../scene --data_factor 1 --result_dir ../../out --max_steps 7000 --save_ply --ply_steps 7000 --disable_viewer
ls -la ../../out ../../out/ply 2>/dev/null

In [ ]:
# the result: download the .ply and drag it into https://playcanvas.com/supersplat/editor
from google.colab import files
plys = sorted(glob.glob("out/ply/*.ply") + glob.glob("out/**/*.ply", recursive=True))
print(plys)
if plys: files.download(plys[-1])